# YOLO Segmentation Example

## YOLO Instance Segmentation

This notebook is an example of instance segmentation with **Ultralytics YOLO**.

```
Image (URL or file) → YOLO segmentor → Annotated image + masks + bounding boxes + confidence scores
```

Unlike detection (bounding box only), segmentation **draws a pixel-level mask** over each detected object, enabling precise shape information.

### Available models

| Model | Params | Speed |
|-------|--------|-------|
| `yolo11x-seg.pt` | 62.1 M | slowest / most accurate |
| `yolo11l-seg.pt` | 27.6 M | — |
| `yolo11m-seg.pt` | 22.4 M | — |
| `yolo11s-seg.pt` | 10.1 M | — |
| `yolo11n-seg.pt` |  2.9 M | fastest / lightest |

Change `MODEL_NAME` in the writefile cell to switch between them.

📄 [Ultralytics YOLO docs](https://docs.ultralytics.com/)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

PROJECT_PATH = "/content/drive/MyDrive/CV-Samples/yolo"
!mkdir -p "{PROJECT_PATH}"
%cd "{PROJECT_PATH}"
!ls

In [ ]:
# Install Ultralytics
!pip install ultralytics -q

import ultralytics
ultralytics.checks()  # check environment

In [ ]:
# Download sample images from GitHub
import os

# Image files to download
IMAGE_FILES = [
    'teefarm-pile-1651945_640.jpg',
]

BASE_URL = 'https://raw.githubusercontent.com/mastnk/cv-samples/main/yolo'
for fname in IMAGE_FILES:
    url = f'{BASE_URL}/{fname}'
    dest = f'{PROJECT_PATH}/{fname}'
    if not os.path.exists(dest):
        os.system(f'wget -q "{url}" -O "{dest}"')
        print(f'Downloaded: {fname}')
    else:
        print(f'Already exists: {fname}')

%cd "{PROJECT_PATH}"
!ls


## Using Your Own Images

There are two ways to provide images for segmentation:

**① Specify a URL**  
Pass a direct image URL to the `--url` flag when running the script:
```
%run segmentation.py --url https://example.com/image.jpg
```

**② Upload images to the `PROJECT_PATH/` folder**  
Place your image files in `PROJECT_PATH/`, then run the script with `--file` or `--dir`.

The easiest way to upload is through **Google Drive**:  
Open [drive.google.com](https://drive.google.com), navigate to `My Drive / CV-Samples / yolo/`, and drag & drop your files there.  
They will be immediately available in Colab via the mounted drive — no extra upload step needed.

```
%run segmentation.py --file your_image.jpg   # single file
%run segmentation.py --dir  .                # all images in folder
```

## Selecting a Model

In the writefile cell below, change `MODEL_NAME` to switch models.  
When multiple `MODEL_NAME = ...` lines are listed, **only the last one takes effect**.

```python
# MODEL_NAME = 'yolo11x-seg.pt'  # 62.1M params — most accurate, slowest
# MODEL_NAME = 'yolo11l-seg.pt'  # 27.6M params
# MODEL_NAME = 'yolo11m-seg.pt'  # 22.4M params
# MODEL_NAME = 'yolo11s-seg.pt'  # 10.1M params
MODEL_NAME   = 'yolo11n-seg.pt'  #  2.9M params — fastest, lightest  ← active
```

Larger models are more accurate but take longer to run. Start with `yolo11n-seg.pt` for quick experiments.

In [ ]:
# Save this cell as a Python file (Execute after editing)
%%writefile segmentation.py
"""YOLO Instance Segmentation — command-line interface.

Usage:
  %run segmentation.py --url  <image_url>  [--disp] [--conf FLOAT]
  %run segmentation.py --file <image_path> [--disp] [--conf FLOAT]
  %run segmentation.py --dir  <image_dir>  [--disp] [--conf FLOAT]
"""

import argparse
import glob
import os

from ultralytics import YOLO
from PIL import Image
import requests
from io import BytesIO

# ---- IPython display (works when executed via %run in Colab) -----
try:
    from IPython.display import display as ipy_display
    _has_ipy = True
except ImportError:
    _has_ipy = False

# ---- Configuration -----------------------------------------------
MODEL_NAME   = 'yolo11n-seg.pt'  #  2.9M params — fastest / lightest
# MODEL_NAME = 'yolo11s-seg.pt'  # 10.1M params
# MODEL_NAME = 'yolo11m-seg.pt'  # 22.4M params
# MODEL_NAME = 'yolo11l-seg.pt'  # 27.6M params
# MODEL_NAME = 'yolo11x-seg.pt'  # 62.1M params — most accurate

PROJECT_PATH = '/content/drive/MyDrive/CV-Samples/yolo'

# ---- Model loading -----------------------------------------------
model = YOLO(MODEL_NAME)

# ---- Display helper ----------------------------------------------
def show(annotated, label: str, disp: bool) -> None:
    """When --disp is active, display the annotated image then print the filename/URL."""
    if disp:
        if _has_ipy:
            ipy_display(annotated)
        print(label)

# ---- Functions ---------------------------------------------------
def segment_url(url: str, conf: float = 0.25, disp: bool = False):
    """Download an image from a URL and segment objects."""
    image     = Image.open(BytesIO(requests.get(url).content)).convert('RGB')
    results   = model.predict(image, conf=conf, verbose=False)
    annotated = Image.fromarray(results[0].plot()[:, :, ::-1])  # BGR -> RGB
    show(annotated, url, disp)
    boxes = results[0].boxes
    names = results[0].names
    for box in boxes:
        cls_id   = int(box.cls[0])
        conf_val = float(box.conf[0])
        print(f'  {names[cls_id]:<20} {conf_val*100:.1f}%')
    return results


def segment_file(path: str, conf: float = 0.25, disp: bool = False):
    """Segment objects in a single local image file."""
    image     = Image.open(path).convert('RGB')
    results   = model.predict(image, conf=conf, verbose=False)
    annotated = Image.fromarray(results[0].plot()[:, :, ::-1])  # BGR -> RGB
    show(annotated, path, disp)
    boxes = results[0].boxes
    names = results[0].names
    for box in boxes:
        cls_id   = int(box.cls[0])
        conf_val = float(box.conf[0])
        print(f'  {names[cls_id]:<20} {conf_val*100:.1f}%')
    return results


def segment_dir(directory: str, conf: float = 0.25, disp: bool = False):
    """Segment objects in all images in a directory."""
    exts = ['jpg', 'jpeg', 'png', 'bmp', 'JPG', 'JPEG', 'PNG', 'BMP']
    filepaths = []
    for ext in exts:
        filepaths += sorted(glob.glob(os.path.join(directory, f'*.{ext}')))

    if not filepaths:
        print(f'No images found in: {directory}')
        return

    for path in filepaths:
        segment_file(path, conf, disp)
        print()


# ---- Argument parsing --------------------------------------------
parser = argparse.ArgumentParser(description='YOLO Instance Segmentation')
group  = parser.add_mutually_exclusive_group(required=True)
group.add_argument('--url',  type=str,              help='Image URL to segment')
group.add_argument('--file', type=str,              help='Path to a single image file')
group.add_argument('--dir',  type=str,              help='Directory of images to segment')
parser.add_argument('--conf', type=float, default=0.25, help='Confidence threshold (default: 0.25)')
parser.add_argument('--disp', action='store_true',      help='Display annotated image and filename/URL before results')
args = parser.parse_args()

# ---- Run ---------------------------------------------------------
if args.url:
    segment_url(args.url,   conf=args.conf, disp=args.disp)
elif args.file:
    segment_file(args.file, conf=args.conf, disp=args.disp)
elif args.dir:
    segment_dir(args.dir,   conf=args.conf, disp=args.disp)


## `segmentation.py` Usage

After running the `%%writefile` cell above, `segmentation.py` is saved in `PROJECT_PATH`.
Run it with **`%run`** (not `!python`) so that inline image display works in Colab.

```
%run segmentation.py --url  <image_url>        # segment objects in a remote image
%run segmentation.py --file <image_path>       # segment objects in a single local file
%run segmentation.py --dir  <image_directory>  # segment objects in all images in a folder
```

**Optional arguments**

| Flag | Default | Description |
|------|---------|-------------|
| `--disp` | off | Display annotated image and filename / URL before results |
| `--conf <f>` | `0.25` | Confidence threshold for detections (0.0 – 1.0) |

**Examples**

```bash
# Segment objects in a remote image and display it
%run segmentation.py --url https://cdn.pixabay.com/photo/2016/12/13/05/15/puppy-1903313_640.jpg --disp

# Segment objects in one file, lower confidence threshold
%run segmentation.py --file street.jpg --conf 0.1 --disp

# Segment objects in every image in the folder, display each one
%run segmentation.py --dir . --disp
```

**Output format (with `--disp`)**

```
<annotated image with masks displayed inline>
street.jpg
  person               87.3%
  car                  91.5%
  bicycle              76.2%
```

## Execution Methods

Use **`%run`** to execute `segmentation.py` inside the Colab kernel — this enables inline image display with `--disp`.

| Mode | Flag | Description |
|------|------|-------------|
| From URL | `--url <url>` | Fetches and segments objects in a remote image |
| Single file | `--file <path>` | Segments objects in one local image |
| Directory | `--dir <path>` | Segments objects in all images in a folder |

Add `--disp` to display each annotated image (with masks) and its filename / URL before the results.  
Add `--conf <f>` to change the confidence threshold (default: 0.25).

In [ ]:
# Execute: segment objects in an image from URL (with display)
%cd "{PROJECT_PATH}"
%run segmentation.py --disp --url https://cdn.pixabay.com/photo/2016/12/13/05/15/puppy-1903313_640.jpg


In [ ]:
# Execute: segment objects in a single local image file (with display)
%cd "{PROJECT_PATH}"
%run segmentation.py --disp --file teefarm-pile-1651945_640.jpg


In [ ]:
# Execute: segment objects in all images in a directory (with display)
%cd "{PROJECT_PATH}"
%run segmentation.py --disp --dir .


💾 **Don't forget to save this notebook!**

To keep your work, go to **File → Save a copy in Drive** before closing.

